# AARS — Agentic Adaptive Retrieval System

[![Open In Colab](https://img.shields.io/badge/Open%20in%20Colab-F9AB00?logo=googlecolab&logoColor=white)](https://colab.research.google.com/github/lekhanpro/aars/blob/main/aars_demo.ipynb)

A metacognitive RAG framework that thinks before retrieving.

In [ ]:
# Install all AARS dependencies
!pip install -q fastapi uvicorn anthropic chromadb sentence-transformers rank-bm25 networkx spacy httpx structlog PyMuPDF pydantic-settings

## Step 1 — Configure
Set your API keys and configure the AARS system. The `ANTHROPIC_API_KEY` is required for the LLM planner, reflection agent, and answer generator.

In [ ]:
import os
import sys

# Set your Anthropic API key (replace with your actual key)
os.environ["ANTHROPIC_API_KEY"] = "sk-ant-REPLACE-ME"

# Ensure the AARS project root is on the Python path
PROJECT_ROOT = os.path.dirname(os.path.abspath("__file__"))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# Import AARS configuration
from config.settings import get_settings, Settings

# Import the LLM client wrapper
from src.llm.client import LLMClient

# Import the pipeline orchestrator
from src.pipeline.orchestrator import PipelineOrchestrator

# Import API schemas for requests and responses
from src.api.schemas import QueryRequest, QueryResponse, RetrievalStrategy

# Load settings from environment / .env file
settings = get_settings()

# Initialize the Anthropic-backed LLM client
llm_client = LLMClient(api_key=settings.anthropic_api_key, settings=settings.llm)

print("Settings loaded successfully.")
print(f"LLM model: {settings.llm.model}")
print(f"Chunk size: {settings.chunker.size}, overlap: {settings.chunker.overlap}")

## Step 2 — Ingest a Document
We'll download the original RAG paper from arXiv and ingest it into our vector store.

In [ ]:
import urllib.request
import asyncio
from io import BytesIO
from fastapi import UploadFile

import chromadb

from src.ingestion.pipeline import IngestionPipeline

# Download the original RAG paper from arXiv
RAG_PAPER_URL = "https://arxiv.org/pdf/2005.11401.pdf"
LOCAL_PATH = "/tmp/rag_paper.pdf"

print("Downloading RAG paper from arXiv...")
urllib.request.urlretrieve(RAG_PAPER_URL, LOCAL_PATH)
print(f"Saved to {LOCAL_PATH}")

# Connect to ChromaDB (ephemeral in-memory client for the demo)
chroma_client = chromadb.Client()

# Create the ingestion pipeline
ingestion = IngestionPipeline(chroma_client=chroma_client, settings=settings)

# Wrap the PDF bytes as a FastAPI UploadFile for the pipeline
with open(LOCAL_PATH, "rb") as f:
    pdf_bytes = f.read()

upload_file = UploadFile(filename="rag_paper.pdf", file=BytesIO(pdf_bytes))

# Run ingestion
result = await ingestion.ingest(file=upload_file, collection="default")

print(f"\nIngestion complete!")
print(f"Documents ingested: {result.documents_ingested}")
print(f"Chunks indexed: {result.chunks_created}")
print(f"Collection: {result.collection}")

## Step 3 — Run a Simple Query
A straightforward factual query. The planner should select **keyword** retrieval since this asks for a specific fact.

In [ ]:
# Build the orchestrator with our in-memory ChromaDB client
orchestrator = PipelineOrchestrator(
    llm_client=llm_client,
    chroma_client=chroma_client,
    settings=settings,
)

# Create a simple factual query
simple_request = QueryRequest(
    query="What is Retrieval-Augmented Generation?",
    enable_reflection=True,
    enable_trace=True,
)

# Run the full AARS pipeline
simple_response = await orchestrator.run(simple_request)

# Display the planner's decision
plan = simple_response.retrieval_plan
print("=== Planner Decision ===")
print(f"  Strategy : {plan.strategy.value}")
print(f"  Query Type : {plan.query_type.value}")
print(f"  Complexity : {plan.complexity.value}")
print(f"  Rewritten : {plan.rewritten_query}")

# Display the answer with confidence
print(f"\n=== Answer (confidence: {simple_response.confidence:.2f}) ===")
print(simple_response.answer)

# Display citations
print(f"\n=== Citations ({len(simple_response.citations)}) ===")
for i, cite in enumerate(simple_response.citations, 1):
    print(f"  [{i}] doc_id={cite.doc_id}")
    print(f"      \"{cite.text[:120]}...\"" if len(cite.text) > 120 else f"      \"{cite.text}\"")

## Step 4 — Multi-hop Query
This query requires connecting information from multiple documents. The planner should select **graph** retrieval and decompose into sub-queries.

In [ ]:
# Create a multi-hop query that requires connecting multiple pieces of information
multihop_request = QueryRequest(
    query=(
        "How does the retrieval component in RAG relate to the generation component, "
        "and what are the key differences from standard language models?"
    ),
    enable_reflection=True,
    enable_trace=True,
)

# Run the full AARS pipeline
multihop_response = await orchestrator.run(multihop_request)

# Display the planner's decision, including decomposed sub-queries
plan = multihop_response.retrieval_plan
print("=== Planner Decision ===")
print(f"  Strategy           : {plan.strategy.value}")
print(f"  Query Type         : {plan.query_type.value}")
print(f"  Complexity         : {plan.complexity.value}")
print(f"  Decomposed Queries :")
for j, sq in enumerate(plan.decomposed_queries, 1):
    print(f"    {j}. {sq}")

# Show how many reflection iterations were needed
print(f"\nReflection iterations: {len(multihop_response.reflection_results)}")

# Display the final answer
print(f"\n=== Answer (confidence: {multihop_response.confidence:.2f}) ===")
print(multihop_response.answer)

## Step 5 — Inspect the Reflection Loop
A deliberately vague query to trigger the reflection agent's retry mechanism.

In [ ]:
# A deliberately vague query to trigger the reflection loop
vague_request = QueryRequest(
    query="Tell me about the thing with the vectors",
    enable_reflection=True,
    enable_trace=True,
)

# Run the pipeline
vague_response = await orchestrator.run(vague_request)

# Inspect each reflection iteration
print("=== Reflection Loop Details ===")
for i, ref in enumerate(vague_response.reflection_results, 1):
    print(f"\n--- Iteration {i} ---")
    print(f"  Sufficient         : {ref.sufficient}")
    print(f"  Confidence         : {ref.confidence:.2f}")
    print(f"  Missing Info       : {ref.missing_information}")
    print(f"  Next Query         : {ref.next_query}")

# Display the final answer
print(f"\n=== Final Answer (confidence: {vague_response.confidence:.2f}) ===")
print(vague_response.answer)

## Benchmark: AARS vs Naive RAG
Compare AARS adaptive retrieval against a naive single-strategy approach on the same queries.

In [ ]:
# Define a set of test questions spanning different types
test_questions = [
    "What datasets were used to evaluate RAG?",
    "How does RAG combine parametric and non-parametric memory?",
    "Compare the RAG-Token and RAG-Sequence models.",
    "What role does the retriever play during fine-tuning?",
    "Explain how RAG could be applied to multi-lingual tasks.",
]

# Run each query through full AARS (adaptive strategy + reflection)
aars_results = []
for q in test_questions:
    req = QueryRequest(query=q, enable_reflection=True, enable_trace=True)
    resp = await orchestrator.run(req)
    aars_results.append(resp)

# Run each query through a naive approach: always vector, no reflection
naive_results = []
for q in test_questions:
    req = QueryRequest(query=q, enable_reflection=False, enable_trace=True)
    resp = await orchestrator.run(req)
    naive_results.append(resp)

# Print a formatted comparison table
header = f"{'Query':<58} | {'AARS Strategy':<16} | {'AARS Conf':>9} | {'Naive Conf':>10}"
print(header)
print("-" * len(header))

for q, aars_r, naive_r in zip(test_questions, aars_results, naive_results):
    aars_strategy = aars_r.retrieval_plan.strategy.value if aars_r.retrieval_plan else "n/a"
    aars_conf = f"{aars_r.confidence:.2f}"
    naive_conf = f"{naive_r.confidence:.2f}"
    # Truncate long queries for display
    q_short = (q[:55] + "...") if len(q) > 55 else q
    print(f"{q_short:<58} | {aars_strategy:<16} | {aars_conf:>9} | {naive_conf:>10}")